In [79]:
%pip install selenium==4.22
%pip install python-dotenv


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [80]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
from dotenv import dotenv_values
import time
import os

In [81]:
COURSE_NAME = "24CSE0208-Data Structures using Object Oriented Programming-2024-CSE-3Sem"
BROWSER = "CHROME"
HEADLESS_MODE = False

# Attempt questions even if they are complete (For development/testing purpose only)
ATTEMPT_EVEN_IF_COMPLETE = False

QUESTIONS_TO_SKIP = ["Form a new number", "FInd the numbers of shifts in Insertion sort"]

In [82]:
from selenium import webdriver
BROWSER_OPTIONS_MAP = {
    "CHROME": webdriver.ChromeOptions,
    "FIREFOX": webdriver.FirefoxOptions,
}
BROWSER_DRIVER_MAP = {
    "CHROME": webdriver.Chrome,
    "FIREFOX": webdriver.Firefox,
}

options = BROWSER_OPTIONS_MAP[BROWSER]()

if HEADLESS_MODE == True:
    options.add_argument("-headless")
else:
    options.add_argument("-detach")

In [83]:
def login_into_testpad():
    global driver, wait
    driver = BROWSER_DRIVER_MAP[BROWSER](options=options)
    wait = WebDriverWait(driver, 10)
    
    driver.get("https://course.testpad.chitkara.edu.in")
    wait.until(EC.frame_to_be_available_and_switch_to_it((By.ID, "loginIframe")))

    testpad_credentials = dotenv_values(".env")

    email_element = driver.find_element(By.ID, "email")
    email_element.send_keys(testpad_credentials["TESTPAD_EMAIL"])

    password_element = driver.find_element(By.ID, "password")
    password_element.send_keys(testpad_credentials["TESTPAD_PASSWORD"])

    submit_button = wait.until(EC.element_to_be_clickable((By.ID, "submit")))
    driver.execute_script("arguments[0].click()", submit_button)

    driver.switch_to.parent_frame()

In [84]:
def wait_for_loader_disappearance(): wait.until(EC.invisibility_of_element_located(
    (By.ID, "loader-container")))

def click_course():
    wait.until(EC.visibility_of_all_elements_located(
        (By.CLASS_NAME, "learning-module-card")))

    learning_module_card = driver.find_element(
        By.XPATH, f"//a[text()='{COURSE_NAME}']").find_element(By.XPATH, "../..")

    learning_module_href = learning_module_card.find_element(
        By.CSS_SELECTOR,
        "[class='icons-link-container tooltip-up'][data-customtooltip='Learning Content']"
    )

    wait_for_loader_disappearance()
    wait.until(EC.element_to_be_clickable(learning_module_href))
    driver.execute_script("arguments[0].click()", learning_module_href)

    """
    Wait for subtopics to show up
    """
    no_of_topics = len(wait.until(EC.presence_of_all_elements_located(
        (By.CLASS_NAME, "course-topics"))))

    if no_of_topics == 0:
        raise ValueError("No course topic showing!")

    print(f"Number of subtopics- {no_of_topics}")

    wait_for_loader_disappearance()

    global original_window
    original_window = driver.current_window_handle

In [85]:
EXTENSION_MAPPING = {
    '.c': "C",
    ".java": "Java",
    ".cpp": "C++",
    ".py": "Python",
    ".sql": "SQL"
}

def complete_coding_question(question_file_path):
    coding_question_file_content = open(question_file_path).read()

    dropdown_language_selected = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, "button[data-id='progLang']")))
    codefile_extension = question_file_path[question_file_path.rindex(
        "."):]
    
    if EXTENSION_MAPPING[codefile_extension] != dropdown_language_selected.text:
        wait_for_loader_disappearance()
        driver.execute_script(
            "arguments[0].click()", dropdown_language_selected)

        programming_languages_ul = WebDriverWait(driver, 3).until(
            EC.visibility_of_any_elements_located(
                (By.CSS_SELECTOR, "ul[class='dropdown-menu inner show']"))
        )
        programming_languages_ul = [item for item in programming_languages_ul if item.is_displayed()][0]
        programming_languages = programming_languages_ul.find_elements(
            By.TAG_NAME, "li")

        for language in programming_languages:
            language_text = language.find_element(By.CSS_SELECTOR, "a > span").text
            if language_text == EXTENSION_MAPPING[codefile_extension]:
                language.click()
                break

    wait.until(
        EC.presence_of_element_located((By.CLASS_NAME, "CodeMirror"))
    )

    driver.execute_script("const cm = document.getElementsByClassName('CodeMirror cm-s-default')[0].CodeMirror;" +
                            f"cm.setValue({coding_question_file_content!r});")
    time.sleep(0.2)
    
    submit_code_button = driver.find_element(By.ID, "executeProgram")
    wait_for_loader_disappearance()
    submit_code_button.click()

    try:
        test_cases = wait.until(EC.visibility_of_all_elements_located(
        (By.CLASS_NAME, "test-cases")))
    except TimeoutException:
        print("Submit button wasn't clicked the first time / There was a problem with selecting correct language. Clicking again")
        driver.execute_script("arguments[0].click()", submit_code_button)
        test_cases = wait.until(EC.visibility_of_all_elements_located(
                (By.CLASS_NAME, "test-cases")))

    test_cases_passed = driver.find_elements(
        By.CSS_SELECTOR, f".test-cases > img[src='/icons/check1.svg']")
    if len(test_cases_passed) != len(test_cases):
        raise ValueError("All test cases did not pass ❌")

In [86]:
def complete_textarea_question(question_file_path):
    mq_question_file_content = open(question_file_path).read()

    questions = wait.until(EC.visibility_of_all_elements_located((
        By.CSS_SELECTOR, "#compileTestcases > [class='col-md-12 test_cases']")))
    answers = mq_question_file_content.strip().split("\n\n\n")
    if len(questions) != len(answers):
        raise ValueError(f"{len(questions)} questions, {len(answers)} answers")

    for i in range(len(questions)):
        driver.execute_script(f"const textarea = document.querySelector('#txtOutput{i}');" + f"textarea.value = {answers[i]!r};")
    time.sleep(0.2)
    
    wait_for_loader_disappearance()
    execute_program_button = driver.find_element(
        By.ID, "executeProgram")
    wait.until(EC.element_to_be_clickable(
        execute_program_button)).click()
    outcomes = WebDriverWait(driver, 5).until(EC.visibility_of_all_elements_located(
        (By.CSS_SELECTOR, "label.test_case_result > img.correct_result")))
    
    no_of_correct_answers = 0
    for outcome in outcomes:
        outcome_image_type = outcome.get_attribute("src")
        if outcome_image_type == "https://course.testpad.chitkara.edu.in" + "/icons/check1.svg":
            no_of_correct_answers += 1     

    if no_of_correct_answers != len(questions):
        raise ValueError(
            f"Only {no_of_correct_answers}/{len(questions)} correct answers ❌\n")

In [87]:
def complete_mcq_question(question_file_path):
    mcq_question_file_content = open(question_file_path).read()

    mcq_options = wait.until(EC.visibility_of_all_elements_located((
        By.CSS_SELECTOR, "[class='col-12 mt-2 div-radio-btn-choose-answer']")))
    mcq_answer = int(mcq_question_file_content)
    wait_for_loader_disappearance()
    mcq_options[mcq_answer -
                1].find_element(By.CLASS_NAME, "choose-answer-checkmark").click()

    submit_button = driver.find_element(By.CSS_SELECTOR, "input#executeMCQ")
    wait_for_loader_disappearance()
    submit_button.click()
    wait.until(EC.element_to_be_clickable(submit_button))

    WebDriverWait(driver, 5).until(EC.visibility_of_element_located(
        (By.CSS_SELECTOR, "span.text-success")))

In [88]:
def complete_tutorial():
    try:
        code_submision_elements = wait.until(EC.visibility_of_all_elements_located((
        By.CLASS_NAME, "code_submision")))
    except:
        wait_for_loader_disappearance()
        code_submision_elements = driver.find_elements(
            By.CLASS_NAME, "code_submision")

    for code_submision in code_submision_elements:
        submission_status = code_submision.find_element(
            By.CSS_SELECTOR, "div[class='col-12 col-md-12 form-group execute_btn'] > span:last-of-type")

        if not submission_status.is_displayed():
            wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "execute_button"))).click()
            wait.until(EC.visibility_of(submission_status))
            time.sleep(1)
            wait_for_loader_disappearance()

In [89]:
def complete_course_subtopic(subtopic_number: str, subtopic_name: str, subtopic_questions: list):
    no_of_subtopic_questions = len(subtopic_questions)
    for i in range(no_of_subtopic_questions):
    # for i in range(0,1):

        question = subtopic_questions[i]

        question_number = question.find_element(
            By.CLASS_NAME, "course-question-number").text
        question_name = question.find_element(
            By.CLASS_NAME, "course-question-name").text
        question_type = question.find_element(
            By.CLASS_NAME, "course-question-type").text

        question_status = bool(question.find_elements(
            By.CSS_SELECTOR, ".question-complete-tick-mark-mv > img"))


        # if question_type == "Coding": continue
        if question_status and (ATTEMPT_EVEN_IF_COMPLETE == False or question_type == "Tutorial"): continue
        if question_name.strip() in QUESTIONS_TO_SKIP:
            print(f"{question_number}. Question skipped- {question_name} ❌")
            continue

        print(f"{question_number}. Attempting question- {question_name} ({question_type})")

        question.find_element(
            By.CLASS_NAME, "isQuestionModal").click()

        wait.until(EC.number_of_windows_to_be(2))
        driver.switch_to.window(driver.window_handles[1])

        if question_type == "Tutorial":
            try:
                complete_tutorial()
            except TimeoutException:
                input(
                    "There was a timeout error while completing question. Press enter to retry.")
                driver.refresh()
                wait_for_loader_disappearance()
                print(
                    f"Question ({question_name} ({question_type})) did not complete on first attempt. Attempting it again.")
                complete_tutorial()

            time.sleep(1)  # For the purpose of visual testing
            driver.close()
            driver.switch_to.window(original_window)
            continue
        
        question_file_path = os.path.join(
            "Testpad-Solutions", COURSE_NAME)

        for folder in os.listdir(question_file_path):
            if folder.startswith(subtopic_number + ". "):
                question_file_path = os.path.join(
                    question_file_path, folder)
                break
            
        for folder in os.listdir(question_file_path):
            if folder.startswith(question_number + ". "):
                question_file_path = os.path.join(
                    question_file_path, folder)
                break
        if question_type != "Coding":
            question_file_path = os.path.join(
                question_file_path, "Answer.txt")
        else:
            for file in os.listdir(question_file_path):
                for e in EXTENSION_MAPPING.keys():
                    if file.endswith(e):
                        question_file_path = os.path.join(
                            question_file_path, file)
                        break
        
        try:
            if question_type == "Coding":
                complete_coding_question(question_file_path)
            elif question_type == "MCQ":
                complete_mcq_question(question_file_path)
            elif question_type == "MQ":
                complete_textarea_question(question_file_path)
            else:
                raise ValueError(f"Unknown question type- {question_type}")
            
            print("Question successfully completed ✅")
        except (TimeoutException, StaleElementReferenceException):
            input("There was a error while completing question. Press enter to retry.")
            time.sleep(2)
            driver.refresh()
            wait_for_loader_disappearance()
            print(f"Question ({question_name} ({question_type})) did not complete on first attempt. Attempting it again.")
            if question_type == "Coding":
                complete_coding_question(question_file_path)
            elif question_type == "MCQ":
                complete_mcq_question(question_file_path)
            elif question_type == "MQ":
                complete_textarea_question(question_file_path)
            print("Question successfully completed ✅")

        time.sleep(1)  # For the purpose of visual testing
        driver.close()
        driver.switch_to.window(original_window)

In [90]:
def complete_course():
    print("Course subtopics-")
    all_topics = driver.find_elements(By.CLASS_NAME, "course-topics")
    for topic in all_topics:
        topic_number = topic.find_element(
            By.CSS_SELECTOR, ".course-topics-number > p"
        ).text
        topic_name = topic.find_element(
            By.CSS_SELECTOR, ".course-topics-name > p"
        ).text
        print(f"{topic_number}) {topic_name}")
    print("\n\n")

    wait_for_loader_disappearance()
    time.sleep(2)  # For visual testing

    # re-fetch topics fresh each iteration
    for i in range(len(driver.find_elements(By.CLASS_NAME, "course-topics"))):
        course_topics = driver.find_elements(By.CLASS_NAME, "course-topics")
        course_topic = course_topics[i]

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});", course_topic
        )

        subtopic_number = course_topic.find_element(
            By.CSS_SELECTOR, "div.course-topics-number > p"
        ).text

        # ✅ FIX: check completion inside this specific topic
        if not ATTEMPT_EVEN_IF_COMPLETE:
            is_subtopic_complete = bool(course_topic.find_elements(By.CLASS_NAME, "icon-1"))
            if is_subtopic_complete:
                print(f"Skipping subtopic {subtopic_number} (already complete ✅)")
                continue

        # Ensure the subtopic is clicked
        if "clicked" not in course_topic.get_attribute("class"):
            wait_for_loader_disappearance()
            course_topic.click()

        wait_for_loader_disappearance()
        subtopic_name = course_topic.find_element(
            By.CSS_SELECTOR, ".course-topics-name > p"
        ).text

        subtopic_questions = [
            q for q in driver.find_elements(By.CLASS_NAME, "course-question") if q.is_displayed()
        ]
        if not subtopic_questions:
            raise ValueError("No subtopic questions! ❌")

        print("#" * 30)
        print(f"{subtopic_number}) Topic name - {subtopic_name} ({len(subtopic_questions)} questions)")

        no_of_incomplete = sum(
            1 for q in subtopic_questions if not q.find_elements(
                By.CSS_SELECTOR, "div.question-complete-tick-mark-mv > img"
            )
        )
        print(f"{no_of_incomplete}/{len(subtopic_questions)} incomplete questions")

        complete_course_subtopic(subtopic_number, subtopic_name, subtopic_questions)

        print("#" * 30, "\n\n")

    print(f"Course {COURSE_NAME} successfully completed 🎉")


In [91]:
try:
    login_into_testpad()
    click_course()
    complete_course()
except Exception as e:
    print(e)
finally:
    input("Press enter to properly close remote browser")
    driver.quit()
    print("Remote browswer closed ✅")

Number of subtopics- 12
Course subtopics-
1) Recursion -1
2) Arrays and Linear Search
3) Binary Search
4) Sorting - 1
5) Sorting - 2
) 
) 
) 
) 
) 
) 
) 



##############################
1) Topic name - Recursion -1 (7 questions)
1/7 incomplete questions
5. Question skipped- Form a new number ❌
############################## 


Skipping subtopic 2 (already complete ✅)
Skipping subtopic 3 (already complete ✅)
Skipping subtopic 4 (already complete ✅)
##############################
5) Topic name - Sorting - 2 (11 questions)
1/11 incomplete questions
7. Question skipped- FInd the numbers of shifts in Insertion sort ❌
############################## 


Skipping subtopic 6 (already complete ✅)
Skipping subtopic 7 (already complete ✅)
Skipping subtopic 8 (already complete ✅)
Skipping subtopic 9 (already complete ✅)
Skipping subtopic 10 (already complete ✅)
Skipping subtopic 11 (already complete ✅)
Skipping subtopic 12 (already complete ✅)
Course 24CSE0208-Data Structures using Object Oriented